In [ ]:
!pip install -q streamlit tensorflow scikit-learn pandas numpy
print("Installed required libraries")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 45.0 MB/s eta 0:00:00
Installed required libraries


In [2]:
from google.colab import files
uploaded = files.upload()  # choose telecom_data.csv

import pandas as pd
dataset_name = list(uploaded.keys())[0]
df = pd.read_csv(dataset_name)
print("✅ Dataset loaded:", dataset_name)
print("Shape:", df.shape)
df.head()


Saving telecom_data.csv to telecom_data.csv
✅ Dataset loaded: telecom_data.csv
Shape: (1000, 20)


,CustomerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST0001,Male,0,No,No,50,Yes,No phone service,DSL,Yes,Yes,No,No,Yes,Month-to-month,Yes,Bank transfer,33.34,7762.99,No
1,CUST0002,Female,0,No,Yes,46,Yes,Yes,Fiber optic,No,No,No,No,No,One year,Yes,Bank transfer,33.30,4435.66,No
2,CUST0003,Male,0,No,Yes,3,Yes,No phone service,DSL,No,No,Yes,Yes,Yes,Month-to-month,No,Mailed check,22.24,3771.49,Yes
3,CUST0004,Male,0,No,Yes,6,Yes,Yes,No,Yes,No,No,No,Yes,Two year,Yes,Bank transfer,24.98,7876.52,No
4,CUST0005,Male,0,Yes,No,34,Yes,No,Fiber optic,Yes,Yes,No,Yes,Yes,Month-to-month,No,Bank transfer,46.37,4477.80,Yes


In [3]:
import numpy as np, pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout


In [4]:
df = df.dropna(subset=['Churn'])

# Encode target
if df['Churn'].dtype == 'object':
    df['Churn'] = df['Churn'].map({'Yes':1,'No':0}).fillna(0).astype(int)

In [5]:
# One-hot encode categorical columns
data_enc = pd.get_dummies(df, drop_first=True)

# Split X, y
y = data_enc['Churn']
X = data_enc.drop('Churn', axis=1)

# Scale numeric columns
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pickle.dump(scaler, open("scaler.pkl","wb"))

In [6]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [7]:
# Build ANN
model = Sequential([
    Dense(64, activation='relu', input_dim=X_train.shape[1]),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("Training model...")
model.fit(X_train, y_train, epochs=15, batch_size=32, validation_split=0.2, verbose=1)


Training model...
Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6610 - loss: 0.7186 - val_accuracy: 0.7812 - val_loss: 0.5615
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7670 - loss: 0.5363 - val_accuracy: 0.7625 - val_loss: 0.5702
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7711 - loss: 0.4818 - val_accuracy: 0.7625 - val_loss: 0.5711
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8516 - loss: 0.3479 - val_accuracy: 0.7625 - val_loss: 0.5885
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8980 - loss: 0.3144 - val_accuracy: 0.7688 - val_loss: 0.5992
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9532 - loss: 0.2076 - val_accuracy: 0.7500 - val_loss: 0.6203
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9575 - loss: 0.1585 - val_accuracy: 0.7188 - val_loss: 0.6446
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9725 - loss: 0.1183 - val_accuracy: 0.7063 - val_loss: 0.6688
Ep

In [8]:
model.save("churn_ann.h5")
print("Model and scaler saved (churn_ann.h5, scaler.pkl)")

Model and scaler saved (churn_ann.h5, scaler.pkl)


In [27]:
app_code = r'''
import streamlit as st
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

st.set_page_config(page_title="Telecom Churn Prediction", layout="centered")
st.title("📡 Telecom Customer Churn Prediction")

# Load model and scaler
model = load_model("churn_ann.h5")
scaler = pickle.load(open("scaler.pkl","rb"))

st.header("Enter Customer Details")

# Columns layout
col1, col2 = st.columns(2)
with col1:
    customer_id = st.text_input("Customer ID")
    gender = st.selectbox("Gender", ["Male","Female"])
    senior_citizen = st.selectbox("Senior Citizen", [0,1])
    dependents = st.selectbox("Dependents", ["Yes","No"])
    tenure = st.number_input("Tenure (months)", 0, 72, 12)
    monthly_charges = st.number_input("Monthly Charges ($)", 0.0, 150.0, 70.0)
    total_charges = st.number_input("Total Charges ($)", 0.0, 8000.0, 1200.0)
    contract = st.selectbox("Contract Type", ["Month-to-month","One year","Two year"])
with col2:
    internet_service = st.selectbox("Internet Service", ["DSL","Fiber optic","No"])
    online_security = st.selectbox("Online Security", ["Yes","No"])
    tech_support = st.selectbox("Tech Support", ["Yes","No"])
    payment_method = st.selectbox("Payment Method", ["Electronic check","Mailed check","Bank transfer","Credit card"])
    phone_service = st.selectbox("Phone Service", ["Yes","No"])
    multiple_lines = st.selectbox("Multiple Lines", ["Yes","No","No phone service"])
    streaming_movies = st.selectbox("Streaming Movies", ["Yes","No","No internet service"])
    streaming_tv = st.selectbox("Streaming TV", ["Yes","No","No internet service"])
    online_backup = st.selectbox("Online Backup", ["Yes","No","No internet service"])
    paperless_billing = st.selectbox("Paperless Billing", ["Yes","No"])

if st.button("Predict Churn"):
    df_input = pd.DataFrame({
        'CustomerID':[customer_id],
        'Gender':[gender],
        'SeniorCitizen':[senior_citizen],
        'Dependents':[dependents],
        'tenure':[tenure],
        'MonthlyCharges':[monthly_charges],
        'TotalCharges':[total_charges],
        'Contract':[contract],
        'InternetService':[internet_service],
        'OnlineSecurity':[online_security],
        'TechSupport':[tech_support],
        'PaymentMethod':[payment_method],
        'PhoneService':[phone_service],
        'MultipleLines':[multiple_lines],
        'StreamingMovies':[streaming_movies],
        'StreamingTV':[streaming_tv],
        'OnlineBackup':[online_backup],
        'PaperlessBilling':[paperless_billing]
    })

    # One-hot encoding for categorical variables
    df_input = pd.get_dummies(df_input)

    # Ensure all features from training scaler exist
    for c in scaler.feature_names_in_:
        if c not in df_input.columns:
            df_input[c] = 0
    df_input = df_input[scaler.feature_names_in_]

    # Scale input
    scaled = scaler.transform(df_input)
    prob = model.predict(scaled)[0][0]

    st.subheader("Prediction Result")
    if prob > 0.5:
        st.error(f"⚠️ High risk of churn ({prob*100:.2f}%)")
    else:
        st.success(f"✅ Low risk of churn ({(1-prob)*100:.2f}%)")
'''

with open("app.py","w") as f:
    f.write(app_code)
print("✅ Streamlit app file created with extended customer details")


✅ Streamlit app file created with extended customer details


In [29]:
print("Launching Streamlit... please wait 25 seconds.")

Launching Streamlit... please wait 25 seconds.


In [30]:
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)